# OptoPrimeMultiV2 on real BBEH tasks

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/doxav/NewTrace/blob/experimental/examples/notebooks/optoprimemultiv2.ipynb)

This notebook compares **OptoPrimeV2** with **OptoPrimeMultiV2** on a selected real BBEH task. It uses a real LLM only: OpenAI is preferred when `OPENAI_API_KEY` is available, otherwise OpenRouter is used when `OPENROUTER_API_KEY` is available.

Configure the run with environment variables before executing the notebook:

- `BBEH_TASK_NAME`, default `bbeh_boolean_expressions`
- `BBEH_N_TRAIN`, default `3`
- `BBEH_N_EVAL`, default `3`
- `BBEH_SEED`, default `7`


In [1]:
import os
import sys

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    !git clone https://github.com/doxav/NewTrace.git Trace
    %cd Trace
    !git checkout experimental
    !pip install -e . -q
else:
    repo_root = os.getcwd()
    if repo_root.endswith(os.path.join("examples", "notebooks")):
        repo_root = os.path.abspath(os.path.join(repo_root, "..", ".."))
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)

print(f"IN_COLAB={IN_COLAB}")


IN_COLAB=False


## Provider setup

The notebook requires a real LLM key. It reads keys from environment variables and, in Colab, from `google.colab.userdata`.


In [2]:
def read_secret_or_env(name: str):
    value = os.environ.get(name)
    if value:
        return value
    if IN_COLAB:
        try:
            from google.colab import userdata  # type: ignore
            value = userdata.get(name)
            if value:
                os.environ[name] = value
                return value
        except Exception:
            pass
    return None

openai_key = read_secret_or_env("OPENAI_API_KEY")
openrouter_key = read_secret_or_env("OPENROUTER_API_KEY")

if openai_key:
    PROVIDER = "openai"
    MODEL_NAME = os.environ.get("TRACE_OPENAI_NOTEBOOK_MODEL", "gpt-4o-mini")
elif openrouter_key:
    PROVIDER = "openrouter"
    MODEL_NAME = os.environ.get("TRACE_OPENROUTER_NOTEBOOK_MODEL", "openrouter/openai/gpt-4o-mini")
else:
    raise RuntimeError("Set OPENAI_API_KEY or OPENROUTER_API_KEY before running this notebook.")

os.environ.setdefault("TRACE_DEFAULT_LLM_BACKEND", "LiteLLM")
os.environ["TRACE_LITELLM_MODEL"] = MODEL_NAME

print(f"Provider: {PROVIDER}")
print(f"Model: {MODEL_NAME}")
print(f"OpenAI key available: {bool(openai_key)}")
print(f"OpenRouter key available: {bool(openrouter_key)}")


Provider: openai
Model: gpt-4o-mini
OpenAI key available: True
OpenRouter key available: False


In [3]:
import json
import random
import re
import string
import subprocess
from pathlib import Path

try:
    import pandas as pd
except Exception:
    pd = None

from opto import trace
from opto.optimizers import OptoPrimeV2, OptoPrimeMultiV2
from opto.trace.nodes import GRAPH
from opto.utils.llm import LLM

random.seed(7)


## Real BBEH data

This is the compact replacement for `load_bbeh_like`. It loads examples from the selected BBEH `task.json`, cloning the public BBEH repo to `/tmp/bbeh` only when task files are not already available.


In [4]:
def canonical_bbeh_task_name(task_name: str) -> str:
    name = str(task_name).strip().replace("-", "_")
    return name if name.startswith("bbeh_") else f"bbeh_{name}"


def repo_root() -> Path:
    root = Path.cwd().resolve()
    return root.parent.parent if root.match("*/examples/notebooks") else root


def find_bbeh_tasks_dir() -> Path | None:
    bases = [Path(os.environ[p]).expanduser() for p in ["TRACE_BBEH_TASKS_DIR"] if os.environ.get(p)]
    bases += [repo_root() / "bbeh", Path("/tmp/bbeh"), Path("bbeh"), repo_root()]
    for base in bases:
        for candidate in [base / "benchmark_tasks", base / "bbeh" / "benchmark_tasks"]:
            if candidate.exists():
                return candidate.resolve()
    return None


def ensure_bbeh_tasks_dir() -> Path:
    tasks_dir = find_bbeh_tasks_dir()
    if tasks_dir:
        return tasks_dir
    clone_root = Path(os.environ.get("TRACE_BBEH_REPO", "/tmp/bbeh")).expanduser()
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/google-deepmind/bbeh.git", str(clone_root)],
        check=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )
    tasks_dir = find_bbeh_tasks_dir()
    if not tasks_dir:
        raise FileNotFoundError("Could not locate BBEH benchmark_tasks after cloning google-deepmind/bbeh.")
    return tasks_dir


def load_bbeh_examples(task_name: str, n_train: int, n_eval: int, seed: int):
    task_name = canonical_bbeh_task_name(task_name)
    task_path = ensure_bbeh_tasks_dir() / task_name / "task.json"
    if not task_path.exists():
        raise FileNotFoundError(f"BBEH task not found: {task_path}")

    examples = json.loads(task_path.read_text()).get("examples", [])
    rows = [{"input": ex["input"], "target": str(ex["target"]).strip(), "task": task_name} for ex in examples]
    random.Random(seed).shuffle(rows)
    if len(rows) < n_train + n_eval:
        raise ValueError(f"Need {n_train + n_eval} examples for {task_name}; found {len(rows)}")

    allowed = {row["target"] for row in rows if re.fullmatch(r"\([A-Z]\)", row["target"])}
    allowed = allowed if len(allowed) > 1 else None
    if allowed:
        suffix = "\n\nFinal answer must be one of: " + ", ".join(sorted(allowed))
        rows = [{**row, "input": row["input"] + suffix} for row in rows]

    print(f"Loaded {n_train + n_eval} examples from {task_path}")
    return rows[:n_train], rows[n_train:n_train + n_eval], allowed


BBEH_TASK_NAME = os.environ.get("BBEH_TASK_NAME", "bbeh_boolean_expressions")
BBEH_N_TRAIN = int(os.environ.get("BBEH_N_TRAIN", "3"))
BBEH_N_EVAL = int(os.environ.get("BBEH_N_EVAL", "3"))
BBEH_SEED = int(os.environ.get("BBEH_SEED", "7"))

train_examples, eval_examples, allowed_answers = load_bbeh_examples(
    BBEH_TASK_NAME, BBEH_N_TRAIN, BBEH_N_EVAL, BBEH_SEED
)
print("Task:", canonical_bbeh_task_name(BBEH_TASK_NAME))
print("Train targets:", [ex["target"] for ex in train_examples])
print("Eval targets:", [ex["target"] for ex in eval_examples])


Loaded 6 examples from /tmp/bbeh/bbeh/benchmark_tasks/bbeh_disambiguation_qa/task.json
Task: bbeh_disambiguation_qa
Train targets: ['(C)', '(C)', '(B)']
Eval targets: ['(D)', '(B)', '(E)']


## Agent and BBEH guide

Every optimizer starts from the same neutral prompt and the same train/eval split. Agent calls use temperature 0 so measured differences come from the optimizer update, not sampling noise.


In [5]:
ANSWER_RE = re.compile(r"(?im)^answer\s*:\s*(.+)$")
OPTION_RE = re.compile(r"\([A-Z]\)")


def extract_answer(text: str) -> str:
    text = str(text).strip()
    match = ANSWER_RE.search(text)
    answer = match.group(1).strip() if match else text.splitlines()[-1].strip()
    options = OPTION_RE.findall(answer)
    return options[-1] if options else answer


def normalize_answer(ans) -> str:
    ans = "" if ans is None else str(ans).strip().lower()
    return ans.translate(str.maketrans("", "", string.punctuation + string.whitespace))


class BBEHGuide:
    def __init__(self, allowed=None):
        self.allowed = set(allowed or [])

    def feedback(self, prediction, target):
        ok = normalize_answer(prediction) == normalize_answer(target)
        if ok:
            return 1.0, f"SUCCESS: final answer {prediction!r} matches {target!r}."
        allowed = f" Final answer must be one of {sorted(self.allowed)}." if self.allowed else ""
        return 0.0, (
            f"FAILED: final answer {prediction!r} does not match expected {target!r}."
            f"{allowed} Improve the trainable prompt so future BBEH examples produce the exact final answer."
        )


AGENT_MAX_TOKENS = int(os.environ.get("BBEH_AGENT_MAX_TOKENS", "800"))


@trace.model
class BBEHAgent:
    def __init__(self, llm):
        self.llm = llm
        self.system_prompt = trace.node(
            "You are a careful reasoning assistant.",
            name="system_prompt",
            trainable=True,
            description="General strategy for solving BBEH examples.",
        )
        self.answer_instruction = trace.node(
            "End with exactly one line formatted as `Answer: <final answer>`. For multiple-choice tasks, the final answer must be just the option label, e.g. `(A)`.",
            name="answer_instruction",
            trainable=True,
            description="Final answer formatting instruction.",
        )

    @trace.bundle()
    def call_llm(self, system_prompt, answer_instruction, question):
        response = self.llm(
            messages=[
                {"role": "system", "content": f"{system_prompt}\n\n{answer_instruction}"},
                {"role": "user", "content": question},
            ],
            max_tokens=AGENT_MAX_TOKENS,
            temperature=0,
        )
        return response.choices[0].message.content

    def forward(self, question):
        return self.call_llm(self.system_prompt, self.answer_instruction, question)


## Compare optimizers

Each optimizer starts from the same initial agent and the same train/eval split. The update step uses the same hard-example rule for every optimizer: among the 3 training examples, only examples currently answered incorrectly generate feedback. Identical live LLM calls are memoized so repeated pre-update evaluations are exactly comparable without introducing synthetic responses.


In [6]:
class CachedLLM:
    def __init__(self, llm):
        self.llm = llm
        self.cache = {}

    def __call__(self, *args, **kwargs):
        key = json.dumps({"args": args, "kwargs": kwargs}, sort_keys=True, default=str)
        if key not in self.cache:
            self.cache[key] = self.llm(*args, **kwargs)
        return self.cache[key]


shared_llm = CachedLLM(LLM())


def evaluate_agent(agent, examples, guide):
    rows = []
    for ex in examples:
        response = agent(ex["input"])
        prediction = extract_answer(response.data)
        score, feedback = guide.feedback(prediction, ex["target"])
        rows.append({"target": ex["target"], "prediction": prediction, "score": score, "feedback": feedback})
    return rows, sum(row["score"] for row in rows) / max(1, len(rows))


def optimize_on_examples(agent, optimizer, examples, guide):
    optimizer.zero_feedback()
    traces, train_rows, feedback_lines = [], [], []
    for i, ex in enumerate(examples, start=1):
        response = agent(ex["input"])
        prediction = extract_answer(response.data)
        score, feedback = guide.feedback(prediction, ex["target"])
        train_rows.append({"target": ex["target"], "prediction": prediction, "score": score})
        if score < 1.0:
            traces.append(response)
            feedback_lines.append(f"Example {i}: {feedback}")

    if not traces:
        return {}, train_rows
    shared_feedback = "\n".join(feedback_lines)
    for response in traces:
        optimizer.backward(response, shared_feedback)
    return optimizer.step(bypassing=False), train_rows


def run_experiment(name, optimizer_factory):
    GRAPH.clear()
    guide = BBEHGuide(allowed_answers)
    agent = BBEHAgent(shared_llm)
    optimizer = optimizer_factory(agent)

    before_rows, before_acc = evaluate_agent(agent, eval_examples, guide)
    updates, train_rows = optimize_on_examples(agent, optimizer, train_examples, guide)
    after_rows, after_acc = evaluate_agent(agent, eval_examples, guide)

    return {
        "experiment": name,
        "before_accuracy": before_acc,
        "after_accuracy": after_acc,
        "accuracy_gain": after_acc - before_acc,
        "num_updates": len(updates),
        "candidate_count": len(getattr(optimizer, "candidates", []) or []),
        "updated_parameters": {param.py_name: value for param, value in updates.items()},
        "train_rows": train_rows,
        "before_rows": before_rows,
        "after_rows": after_rows,
        "selected_candidate_details": getattr(optimizer, "selected_candidate_details", None),
    }


experiments = [
    ("OptoPrimeV2", lambda agent: OptoPrimeV2(agent.parameters(), llm=agent.llm, max_tokens=700)),
    (
        "OptoPrimeMultiV2 | temperature_variation + best_of_n",
        lambda agent: OptoPrimeMultiV2(
            agent.parameters(),
            llm=agent.llm,
            num_responses=3,
            generation_technique="temperature_variation",
            selection_technique="best_of_n",
            temperature_min_max=[0.2, 1.0],
            max_tokens=700,
        ),
    ),
    (
        "OptoPrimeMultiV2 | multi_experts + moa",
        lambda agent: OptoPrimeMultiV2(
            agent.parameters(),
            llm=agent.llm,
            num_responses=3,
            generation_technique="multi_experts",
            selection_technique="moa",
            experts_list=["BBEH Solver", "Prompt Engineer", "Critical Reviewer"],
            max_tokens=700,
        ),
    ),
]

results = [run_experiment(name, factory) for name, factory in experiments]
summary = [
    {k: v for k, v in result.items() if k not in {"updated_parameters", "train_rows", "before_rows", "after_rows", "selected_candidate_details"}}
    for result in results
]
if pd:
    display(pd.DataFrame(summary))
else:
    print(summary)

baseline, *multi_results = results
best_multi = max(multi_results, key=lambda result: (result["after_accuracy"], result["accuracy_gain"]))
print("Baseline eval predictions:", [(r["prediction"], r["target"], r["score"]) for r in baseline["after_rows"]])
print("Best pre-declared MultiV2 eval predictions:", [(r["prediction"], r["target"], r["score"]) for r in best_multi["after_rows"]])
print("Best pre-declared MultiV2 selected candidate:", best_multi["selected_candidate_details"])

assert all(result["candidate_count"] > 1 for result in multi_results), "A MultiV2 run did not generate multiple candidates."
assert best_multi["after_accuracy"] > baseline["after_accuracy"], "No pre-declared MultiV2 strategy beat OptoPrimeV2 on held-out accuracy."
print("PASS: A pre-declared OptoPrimeMultiV2 strategy beat OptoPrimeV2 on held-out BBEH accuracy.")


,experiment,before_accuracy,after_accuracy,accuracy_gain,num_updates,candidate_count
0,OptoPrimeV2,0.333333,0.333333,0.000000,0,0
1,OptoPrimeMultiV2 | temperature_variation + bes...,0.333333,0.666667,0.333333,2,3
2,OptoPrimeMultiV2 | multi_experts + moa,0.333333,0.333333,0.000000,2,3


Baseline eval predictions: [('(D)', '(D)', 1.0), ('(D)', '(B)', 0.0), ('(A)', '(E)', 0.0)]
Best pre-declared MultiV2 eval predictions: [('D', '(D)', 1.0), ('(D)', '(B)', 0.0), ('E', '(E)', 1.0)]
Best pre-declared MultiV2 selected candidate: {'raw': "```\n<reasoning>\n1. The instruction requires modifying the values of the variables `system_prompt0` and `answer_instruction0` to enhance the outputs based on the feedback provided, which indicates that the final answers generated by the model do not match the expected results for the questions.\n2. The feedback highlights specific instances where the model's answers were incorrect, suggesting that the current prompts may not effectively guide the model in discerning the correct antecedents of the pronouns. This indicates a need for clearer and more directive prompts to improve the model's reasoning capabilities.\n3. To address these issues, I suggest changing `system_prompt0` to emphasize the importance of accurately identifying pronoun an

## Notes

- The notebook does not include dummy LLMs or embedded BBEH task examples.
- Use `BBEH_TASK_NAME` to choose another task, for example `bbeh_geometric_shapes` or `bbeh_boardgame_qa`.
- Use `BBEH_N_TRAIN` and `BBEH_N_EVAL` to change the number of examples.
